In [ ]:
"""
====================================================================
  ♜ HFT VECTOR BUNDLE ENGINE — Jim Simons Inspired Market Geometry
  ------------------------------------------------------------------

  IDEA:
  -----
  Instead of only using price indicators like UTBot,
  we create a "market manifold" of multiple dimensions:

    • Price acceleration
    • Volume anomalies
    • Volatility curvature
    • Correlation flow
    • Mean reversion pressure
    • Micro trend persistence
    • Liquidity shock
    • Relative strength
    • Statistical deformation

  These features form a VECTOR FIELD.

  Then:
    → We compute a VECTOR BUNDLE SCORE
    → This score estimates probability that HFTs are accumulating
    → Strategy trades only high probability regions

  COMBINED STRATEGIES:
    1. Pure HFT Vector Bundle
    2. Pure UTBot
    3. Hybrid HFT + UTBot

  OUTPUT:
    ✔ Win rate
    ✔ Sharpe ratio
    ✔ Final capital
    ✔ CAGR
    ✔ Drawdown
    ✔ Equity curves
    ✔ Excel report
====================================================================
"""

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================

INITIAL_CAPITAL = 100000

STOP_LOSS_PCT = 0.05
TAKE_PROFIT_PCT = 0.15

MAX_POSITIONS = 10

LOOKBACK_PERIOD = "10y"

ATR_PERIOD = 14

# VECTOR BUNDLE PARAMETERS
VECTOR_WINDOW = 20
HFT_THRESHOLD = 0.70

# UTBOT
UT_ATR_MULT = 1.5

# ============================================================
# LOAD NSE TICKERS
# ============================================================

def load_nse_tickers():

    df = pd.read_csv("data/EQUITY_L.csv")

    symbols = (
        df["SYMBOL"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    return [s + ".NS" for s in symbols if "&" not in s]

# ============================================================
# UTBOT
# ============================================================

def compute_utbot(df):

    df = df.copy()

    tr = np.maximum(
        df["High"] - df["Low"],
        np.maximum(
            abs(df["High"] - df["Close"].shift()),
            abs(df["Low"] - df["Close"].shift())
        )
    )

    atr = tr.rolling(ATR_PERIOD).mean()

    upper = df["Close"] - UT_ATR_MULT * atr
    lower = df["Close"] + UT_ATR_MULT * atr

    trend = [1]

    for i in range(1, len(df)):

        if df["Close"].iloc[i] > lower.iloc[i-1]:
            trend.append(1)

        elif df["Close"].iloc[i] < upper.iloc[i-1]:
            trend.append(-1)

        else:
            trend.append(trend[-1])

    df["trend"] = trend

    df["buy_signal"] = (
        (df["trend"] == 1)
        &
        (df["trend"].shift() == -1)
    )

    df["sell_signal"] = (
        (df["trend"] == -1)
        &
        (df["trend"].shift() == 1)
    )

    return df

# ============================================================
# VECTOR BUNDLE / HFT SCORE
# ============================================================

def compute_hft_score(df):

    df = df.copy()

    returns = df["Close"].pct_change()

    # --------------------------------------------------------
    # 1. PRICE ACCELERATION
    # --------------------------------------------------------

    velocity = returns.rolling(5).mean()

    acceleration = velocity.diff()

    accel_score = (
        acceleration
        .rolling(VECTOR_WINDOW)
        .rank(pct=True)
    )

    # --------------------------------------------------------
    # 2. VOLUME ANOMALY
    # --------------------------------------------------------

    volume_ratio = (
        df["Volume"]
        /
        df["Volume"].rolling(20).mean()
    )

    volume_score = volume_ratio.rank(pct=True)

    # --------------------------------------------------------
    # 3. VOLATILITY COMPRESSION
    # --------------------------------------------------------

    vol = returns.rolling(20).std()

    vol_score = 1 - vol.rank(pct=True)

    # --------------------------------------------------------
    # 4. TREND PERSISTENCE
    # --------------------------------------------------------

    persistence = (
        returns.gt(0)
        .rolling(10)
        .mean()
    )

    persistence_score = persistence.rank(pct=True)

    # --------------------------------------------------------
    # 5. LIQUIDITY SHOCK
    # --------------------------------------------------------

    liquidity = (
        abs(returns)
        /
        np.log1p(df["Volume"])
    )

    liquidity_score = (
        1 - liquidity.rank(pct=True)
    )

    # --------------------------------------------------------
    # 6. CURVATURE
    # --------------------------------------------------------

    curvature = acceleration.diff()

    curvature_score = (
        curvature.abs()
        .rank(pct=True)
    )

    # --------------------------------------------------------
    # 7. RELATIVE STRENGTH
    # --------------------------------------------------------

    rs = (
        df["Close"]
        /
        df["Close"].rolling(50).mean()
    )

    rs_score = rs.rank(pct=True)

    # --------------------------------------------------------
    # COMBINE VECTOR FIELDS
    # --------------------------------------------------------

    df["HFT_SCORE"] = (

        0.20 * accel_score +

        0.20 * volume_score +

        0.15 * vol_score +

        0.15 * persistence_score +

        0.10 * liquidity_score +

        0.10 * curvature_score +

        0.10 * rs_score

    )

    # NORMALIZE
    df["HFT_SCORE"] = (
        df["HFT_SCORE"]
        .clip(0, 1)
    )

    return df

# ============================================================
# LOAD STOCK DATA
# ============================================================

def load_data(tickers):

    all_data = {}

    print("\nDownloading data...\n")

    for i, ticker in enumerate(tickers):

        try:

            df = yf.download(
                ticker,
                period=LOOKBACK_PERIOD,
                auto_adjust=True,
                progress=False
            )

            if len(df) < 200:
                continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df.dropna(inplace=True)

            df = compute_utbot(df)

            df = compute_hft_score(df)

            all_data[ticker] = df

            if i % 50 == 0:
                print(f"Loaded {i} stocks")

        except:
            pass

    return all_data

# ============================================================
# BACKTEST ENGINE
# ============================================================

def backtest_strategy(all_data, mode="HFT"):

    cash = INITIAL_CAPITAL

    positions = {}

    trade_log = []

    equity_curve = []

    date_index = []

    all_dates = sorted(
        set(
            d
            for df in all_data.values()
            for d in df.index
        )
    )

    for current_date in all_dates:

        # ====================================================
        # EXITS
        # ====================================================

        for ticker in list(positions.keys()):

            df = all_data[ticker]

            if current_date not in df.index:
                continue

            row = df.loc[current_date]

            pos = positions[ticker]

            current_price = float(row["Close"])

            stop_price = (
                pos["entry_price"]
                *
                (1 - STOP_LOSS_PCT)
            )

            tp_price = (
                pos["entry_price"]
                *
                (1 + TAKE_PROFIT_PCT)
            )

            exit_trade = False

            if current_price <= stop_price:
                exit_trade = True
                reason = "STOP LOSS"

            elif current_price >= tp_price:
                exit_trade = True
                reason = "TAKE PROFIT"

            elif row["sell_signal"]:
                exit_trade = True
                reason = "SELL SIGNAL"

            if exit_trade:

                proceeds = (
                    pos["shares"]
                    * current_price
                )

                profit = (
                    proceeds
                    - pos["invested"]
                )

                cash += proceeds

                trade_log.append({

                    "Stock": ticker,

                    "Entry Date": pos["entry_date"],

                    "Exit Date": current_date,

                    "Entry Price": pos["entry_price"],

                    "Exit Price": current_price,

                    "Profit": profit,

                    "Return %": (
                        profit
                        /
                        pos["invested"]
                    ) * 100,

                    "Strategy": mode,

                    "Exit Reason": reason
                })

                del positions[ticker]

        # ====================================================
        # ENTRIES
        # ====================================================

        available = (
            MAX_POSITIONS
            - len(positions)
        )

        if available > 0:

            candidates = []

            for ticker, df in all_data.items():

                if ticker in positions:
                    continue

                if current_date not in df.index:
                    continue

                row = df.loc[current_date]

                hft_signal = (
                    row["HFT_SCORE"]
                    >= HFT_THRESHOLD
                )

                ut_signal = bool(row["buy_signal"])

                # --------------------------------------------
                # MODES
                # --------------------------------------------

                if mode == "HFT":

                    entry = hft_signal

                elif mode == "UTBOT":

                    entry = ut_signal

                else:
                    # HYBRID
                    entry = (
                        hft_signal
                        and ut_signal
                    )

                if entry:

                    score = (
                        row["HFT_SCORE"]
                    )

                    candidates.append(
                        (ticker, score, row)
                    )

            candidates.sort(
                key=lambda x: x[1],
                reverse=True
            )

            for ticker, score, row in candidates[:available]:

                allocation = (
                    cash
                    /
                    available
                )

                if allocation <= 0:
                    continue

                shares = (
                    allocation
                    /
                    row["Close"]
                )

                cash -= allocation

                positions[ticker] = {

                    "entry_date": current_date,

                    "entry_price": float(row["Close"]),

                    "shares": shares,

                    "invested": allocation
                }

        # ====================================================
        # EQUITY
        # ====================================================

        pv = cash

        for ticker, pos in positions.items():

            df = all_data[ticker]

            if current_date in df.index:

                pv += (
                    pos["shares"]
                    *
                    float(df.loc[current_date]["Close"])
                )

        equity_curve.append(pv)

        date_index.append(current_date)

    trades = pd.DataFrame(trade_log)

    return trades, equity_curve, date_index

# ============================================================
# PERFORMANCE
# ============================================================

def evaluate_strategy(name, trades, equity_curve):

    if len(equity_curve) == 0:
        return None

    eq = np.array(equity_curve)

    final_capital = eq[-1]

    total_return = (
        final_capital
        /
        INITIAL_CAPITAL
        - 1
    ) * 100

    peak = np.maximum.accumulate(eq)

    drawdown = (
        (eq - peak)
        /
        peak
    )

    max_dd = drawdown.min() * 100

    returns = pd.Series(eq).pct_change().dropna()

    sharpe = 0

    if returns.std() > 0:

        sharpe = (
            returns.mean()
            /
            returns.std()
        ) * np.sqrt(252)

    if len(trades) > 0:

        win_rate = (
            len(trades[trades["Profit"] > 0])
            /
            len(trades)
        ) * 100

    else:
        win_rate = 0

    return {

        "Strategy": name,

        "Final Capital": round(final_capital, 2),

        "Return %": round(total_return, 2),

        "Sharpe": round(sharpe, 2),

        "Max Drawdown %": round(max_dd, 2),

        "Win Rate %": round(win_rate, 2),

        "Trades": len(trades)
    }

# ============================================================
# PLOT
# ============================================================

def plot_results(results):

    plt.figure(figsize=(15, 7))

    for name, curve in results.items():

        plt.plot(
            curve,
            label=name
        )

    plt.title(
        "HFT VECTOR BUNDLE vs UTBOT"
    )

    plt.xlabel("Days")

    plt.ylabel("Portfolio Value")

    plt.legend()

    plt.grid(True)

    plt.show()

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    print("\nLoading NSE stocks...\n")

    tickers = load_nse_tickers()

    tickers = tickers[:300]

    all_data = load_data(tickers)

    # ========================================================
    # RUN STRATEGIES
    # ========================================================

    print("\nRunning HFT Strategy...\n")

    hft_trades, hft_curve, hft_dates = backtest_strategy(
        all_data,
        mode="HFT"
    )

    print("\nRunning UTBOT Strategy...\n")

    ut_trades, ut_curve, ut_dates = backtest_strategy(
        all_data,
        mode="UTBOT"
    )

    print("\nRunning HYBRID Strategy...\n")

    hybrid_trades, hybrid_curve, hybrid_dates = backtest_strategy(
        all_data,
        mode="HYBRID"
    )

    # ========================================================
    # EVALUATION
    # ========================================================

    results = []

    results.append(
        evaluate_strategy(
            "HFT",
            hft_trades,
            hft_curve
        )
    )

    results.append(
        evaluate_strategy(
            "UTBOT",
            ut_trades,
            ut_curve
        )
    )

    results.append(
        evaluate_strategy(
            "HYBRID",
            hybrid_trades,
            hybrid_curve
        )
    )

    results_df = pd.DataFrame(results)

    print("\n================ RESULTS ================\n")

    print(results_df)

    # ========================================================
    # SAVE EXCEL
    # ========================================================

    with pd.ExcelWriter(
        "hft_vector_bundle_results.xlsx"
    ) as writer:

        results_df.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )

        hft_trades.to_excel(
            writer,
            sheet_name="HFT Trades",
            index=False
        )

        ut_trades.to_excel(
            writer,
            sheet_name="UTBOT Trades",
            index=False
        )

        hybrid_trades.to_excel(
            writer,
            sheet_name="HYBRID Trades",
            index=False
        )

    print(
        "\nSaved: hft_vector_bundle_results.xlsx"
    )

    # ========================================================
    # PLOT
    # ========================================================

    curves = {

        "HFT": hft_curve,

        "UTBOT": ut_curve,

        "HYBRID": hybrid_curve
    }

    plot_results(curves)